# 03-area: stage 1 on the full area

Stage 1 runs the chain on the whole area with the code in `src/catchment/`. This first section is the **snapping fix**
(#14): the before and after, on the slice and on the area. The next section (#20) classifies the pairs that still ride out
and back and tries two variants of the second-road rule, alone and together. The area demand map and the full-area solve
time (#15) follow, then the **spot-check routes** (#16) and the **spot-check error** (#18).

Stage 0.5 found about one short pair in seven riding more than 300 m out and back at one end, from snapping onto the far
carriageway of a divided road. `catchment-build reach` now snaps each candidate site and each hex centre to two roads
and keeps the fastest of the four times for a pair; the 17-min cap applies after the minimum. This notebook reads what
`catchment-build` wrote (the after), rebuilds the one-snap table in the same session (the before), and compares them.

In [ ]:
import os
import resource
import tempfile
import time
from collections import defaultdict
from collections.abc import Callable, Iterable
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any

import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import numpy as np
import ortools
import pandas as pd
from ortools.sat.python import cp_model
from pyproj import Transformer
from shapely.geometry import Polygon, box
from shapely.geometry.base import BaseGeometry

from catchment import assumptions
from catchment.area import load_area
from catchment.fetch import DATA
from catchment.assumptions import (BRAND_COUNTS, CAPACITY, MULTIPLIER, N_AREA, ORDERS_PER_STORE_DAY, PICK_AND_PACK,
                                   PROMISE, SCALE, SEED, SOLVER_SEED, TIME_LIMIT, WORKERS)
from catchment.reach import OsrmRouter, build_reach, n_top, served_osrm

FIGS = DATA.parent / "notebooks" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)
SOLVE_BAR_S = 10   # a plan for the default scenario must return within this many seconds
SOLVE_LIMIT_S = 9   # the tighter model's budget for build + solve; CP-SAT gets what the build leaves, so the total stays under SOLVE_BAR_S
MAX_MIN = assumptions.MAX_PAIR_MIN
DEFAULT_MIN = (assumptions.PROMISE - assumptions.PICK_AND_PACK) / assumptions.MULTIPLIER   # 5.0 free-flow minutes
area = load_area()
slice_poly = box(77.005, 28.405, 77.105, 28.480).intersection(area)   # the 0.5 slice
demand = pd.read_parquet(DATA / "demand-r9.parquet")
cand_cmd = pd.read_parquet(DATA / "candidates-r9-x30.parquet")        # catchment-build's outputs (the after)
table_cmd = pd.read_parquet(DATA / "table-r9-x30.parquet")
pts = gpd.read_parquet(DATA / "overture_places.parquet")
places = pd.DataFrame({"lon": pts.geometry.x, "lat": pts.geometry.y, "operating_status": pts.operating_status})   # as build.py loads them
to_utm = Transformer.from_crs(4326, 32643, always_xy=True)   # metres, for distances along the route

## Assumptions

One row of `assumptions.table()` is new in this section. **Assumption** means a labelled choice with no external
source.

In [ ]:
rows = assumptions.table()
pd.set_option("display.max_colwidth", None)
display(rows[rows.iloc[:, 0] == "Second road for snapping"].style.hide(axis="index").set_properties(**{"text-align": "left"}))

## Before and after, measured in one session

Inside one `served_osrm` block the table is built twice with the same function: `snaps=1` (the nearest road only, the
#13 build) and `snaps=2` (the fix). The two-snap table must equal the one `catchment-build` wrote; that check is what
lets the rest of the notebook read the command's output. The block also snaps every candidate site and hex centre to two
roads (to count how many got a distinct second road) and runs the out-and-back sample from the slice section below.

In [ ]:
def route_pair(url: str, src: tuple[float, float], dst: tuple[float, float]) -> tuple[float, float, float]:
    """One /route between two snapped points: minutes, path m while within 100 m of the start, path m after first
    reaching 100 m of the end (the 01-reach method). > 300 m at an end counts as out-and-back."""
    import requests
    rt = requests.get(f"{url}/route/v1/driving/{src[0]:.6f},{src[1]:.6f};{dst[0]:.6f},{dst[1]:.6f}",
                      params={"overview": "full", "geometries": "geojson"}, timeout=300).json()
    assert rt["code"] == "Ok", rt
    (ax, ay), (bx, by) = (to_utm.transform(*w["location"]) for w in rt["waypoints"])
    x, y = to_utm.transform(*np.array(rt["routes"][0]["geometry"]["coordinates"]).T)
    along = np.r_[0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]
    leave = along[np.where(np.hypot(x - ax, y - ay) <= 100)[0].max()]
    arrive = along[-1] - along[np.where(np.hypot(x - bx, y - by) <= 100)[0].min()]
    return rt["routes"][0]["duration"] / 60, leave, arrive

def in_poly(df: pd.DataFrame, poly: BaseGeometry) -> pd.Series:
    return gpd.GeoSeries(gpd.points_from_xy(df.lon, df.lat), index=df.index).within(poly)

def timed(snaps: int, router: OsrmRouter) -> tuple[pd.DataFrame, pd.DataFrame, float]:
    t0 = time.perf_counter()
    c, t = build_reach(area, demand, places, 9, 30, router, snaps=snaps)
    return c, t, time.perf_counter() - t0

area_hexes = demand[in_poly(demand, area)]
slice_hexes = area_hexes[in_poly(area_hexes, slice_poly)]
slice_cands = cand_cmd[in_poly(cand_cmd, slice_poly)]

with served_osrm(DATA) as router:
    cand_before, table_before, secs_before = timed(1, router)
    cand_after, table_after, secs_after = timed(2, router)
    c_lon, c_lat, _ = router.snap(cand_cmd.lon.to_numpy(), cand_cmd.lat.to_numpy(), k=2)
    h_lon, h_lat, _ = router.snap(area_hexes.lon.to_numpy(), area_hexes.lat.to_numpy(), k=2)
    # out-and-back sample: 200 slice pairs with BEFORE minutes <= 6, candidate != hex, seed 42
    in_slice = table_before.candidate.isin(slice_cands.index) & table_before.hex.isin(slice_hexes.index)
    short = table_before[in_slice & (table_before.minutes <= 6) & (table_before.candidate != table_before.hex)]
    c_pos = {h: i for i, h in enumerate(cand_cmd.index)}
    h_pos = {h: i for i, h in enumerate(area_hexes.index)}
    samples = []
    for _, p in short.sample(200, random_state=42).iterrows():
        i, j = c_pos[p.candidate], h_pos[p.hex]
        c_pts = [(c_lon[i, a], c_lat[i, a]) for a in range(2)]
        h_pts = [(h_lon[j, b], h_lat[j, b]) for b in range(2)]
        before = route_pair(router.url, c_pts[0], h_pts[0])
        after = min((route_pair(router.url, c, h) for c in c_pts for h in h_pts), key=lambda r: r[0])
        samples.append({"candidate": p.candidate, "hex": p.hex, "table min before": p.minutes,
                        "before min": before[0], "before leave m": before[1], "before arrive m": before[2],
                        "after min": after[0], "after leave m": after[1], "after arrive m": after[2]})
peak_bytes = router.peak_bytes   # osrm-routed's peak over both builds, the snaps and the sample; read when the block stopped
samples = pd.DataFrame(samples)

# the notebook must read what the command builds
assert list(cand_after.index) == list(cand_cmd.index)
key = ["candidate", "hex"]
a, b = table_after.sort_values(key).reset_index(drop=True), table_cmd.sort_values(key).reset_index(drop=True)
assert len(a) == len(b) and (a[key] == b[key]).all().all() and np.allclose(a.minutes, b.minutes, atol=1e-4), "snaps=2 differs from catchment-build"
print("snaps=2 table equals catchment-build's:", len(a), "pairs")

### Full area

Pairs, table size on disk, build time and memory, before and after. The before table is written to a temporary file to
measure its size; the after size is the file `catchment-build` wrote. The times are `build_reach` alone (OSRM already
serving), so a little under `catchment-build`'s wall time.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    before_path = Path(tmp) / "table-before.parquet"
    table_before.to_parquet(before_path)
    size_before = before_path.stat().st_size
size_after = (DATA / "table-r9-x30.parquet").stat().st_size
full = pd.DataFrame({
    "before (1 snap)": [len(table_before), (table_before.minutes <= DEFAULT_MIN).sum(), size_before / 1e6, secs_before],
    "after (2 snaps)": [len(table_after), (table_after.minutes <= DEFAULT_MIN).sum(), size_after / 1e6, secs_after],
}, index=[f"pairs within {MAX_MIN} min", f"pairs within {DEFAULT_MIN:.1f} min (default scenario)", "table on disk, MB", "build_reach time, s"])
full["change"] = full["after (2 snaps)"] / full["before (1 snap)"] - 1
display(full.round(2))
print(f"osrm-routed peak (both builds, snaps and sample): {peak_bytes / 1e6:,.0f} MB")
print(f"this notebook process's peak RSS: {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:,.0f} MB (holds both tables and the places)")

# second road: how many points got a snap point different from their nearest road's
def distinct(lon: np.ndarray, lat: np.ndarray) -> np.ndarray:
    return ((lon[:, 0] != lon[:, 1]) | (lat[:, 0] != lat[:, 1]))
d_c, d_h = distinct(c_lon, c_lat), distinct(h_lon, h_lat)
print(f"distinct second road: {d_c.sum():,} of {len(d_c):,} candidate sites ({d_c.mean():.1%}), "
      f"{d_h.sum():,} of {len(d_h):,} hex centres ({d_h.mean():.1%})")

### Slice

The same slice as 0.5. Pairs whose candidate site and hex both have centres in the slice, outer-joined before and after
on (candidate, hex). A pair moved if its time changed by more than 0.01 min; the histogram is over moved pairs that were
already within 17 min before.

In [ ]:
def slice_only(t: pd.DataFrame) -> pd.DataFrame:
    return t[t.candidate.isin(slice_cands.index) & t.hex.isin(slice_hexes.index)]

j = slice_only(table_before).merge(slice_only(table_after), on=key, how="outer", suffixes=("_b", "_a"))
both = j.dropna(subset=["minutes_b", "minutes_a"])
saved = both.minutes_b - both.minutes_a
moved = saved[saved.abs() > 0.01]
faster, slower = moved[moved > 0], moved[moved < 0]
new_17 = j.minutes_b.isna().sum()
new_5 = ((j.minutes_a <= DEFAULT_MIN) & ~(j.minutes_b <= DEFAULT_MIN)).sum()
print(f"slice pairs: {len(slice_only(table_before)):,} before, {len(slice_only(table_after)):,} after; only before: {j.minutes_a.isna().sum()}")
print(f"moved (|d| > 0.01 min): {len(moved):,} of {len(both):,} ({len(moved) / len(both):.1%}); slower: {len(slower)} (worst {slower.min() if len(slower) else 0:.3f} min)")
print(f"minutes saved over faster pairs: median {faster.median():.2f}, 90th pct {faster.quantile(.9):.2f}, max {faster.max():.2f}")
print(f"faster by more than 0.5 min: {(moved > 0.5).sum():,} ({(moved > 0.5).sum() / len(both):.1%}); by more than 1 min: {(moved > 1).sum():,} ({(moved > 1).sum() / len(both):.1%})")
print(f"newly within {MAX_MIN} min: {new_17:,}; newly within {DEFAULT_MIN:.1f} min: {new_5:,}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(faster, bins=40, color="#3b6ea5")
ax.set_xlabel("minutes saved by the fastest of four (before - after)")
ax.set_ylabel("pairs")
ax.set_title(f"Slice: {len(faster):,} pairs got faster ({len(faster) / len(both):.1%} of {len(both):,})")
fig.tight_layout()
fig.savefig(FIGS / "03-snapping.png", dpi=150)
plt.show()

![Minutes saved on the slice's moved pairs](figures/03-snapping.png)

### Out-and-back on short pairs

200 slice pairs (seed 42) with free-flow time at most 6 min before and candidate site different from hex. Before: one
`/route` from the candidate's nearest snap to the hex's nearest snap. After: the fastest of the four `/route` calls
between the two snaps of each end. Path while still within 100 m of the start, and after first reaching 100 m of the
end, as in 01-reach; more than 300 m counts as out-and-back.

In [ ]:
for s in ("before", "after"):
    samples[f"{s} path m"] = samples[[f"{s} leave m", f"{s} arrive m"]].max(axis=1)
rows = {}
for s in ("before", "after"):
    rows[s] = {"pairs": len(samples),
               "> 300 m at either end": int((samples[f"{s} path m"] > 300).sum()),
               "start end (candidate)": int((samples[f"{s} leave m"] > 300).sum()),
               "hex end": int((samples[f"{s} arrive m"] > 300).sum()),
               "median start/end path m": samples[f"{s} path m"].median()}
display(pd.DataFrame(rows).round(0))
print("share > 300 m: before", f"{(samples['before path m'] > 300).mean():.1%}", "after", f"{(samples['after path m'] > 300).mean():.1%}")
print(f"max |before route min - table min|: {(samples['before min'] - samples['table min before']).abs().max():.3f}; "
      f"pairs where after is slower than before: {int((samples['after min'] > samples['before min'] + 0.01).sum())}")
print(f"still > 300 m after: {int((samples['after path m'] > 300).sum())}; of them, fixed in minutes (saved > 0.01): "
      f"{int(((samples['after path m'] > 300) & (samples['before min'] - samples['after min'] > 0.01)).sum())}")

Results from the run on 2026-10-29 (copied here because outputs are stripped on commit; re-running updates the printed
tables, not this one). The two-snap table built here equals `catchment-build`'s (1,187,469 pairs, same minutes).

**Full area** (res 9, X 30, 1,458 candidate sites, 4,859 hexes):

| | before (1 snap) | after (2 snaps) | change |
|---|---:|---:|---:|
| pairs within 17 min | 1,137,695 | 1,187,469 | +4.4% |
| pairs within 5.0 min (default scenario) | 50,162 | 59,855 | +19.3% |
| table on disk | 3.69 MB | 3.82 MB | +3.5% |
| `build_reach` time (OSRM already serving) | 59 s | 159 s | 2.7 times |
| `osrm-routed` peak memory (both builds, snaps and sample) | | 410 MB | |

The notebook process peaked at 446 MB (it holds both tables and the places); `catchment-build reach` itself peaked at
505 MB and took 179 s wall time, OSRM serve included. The build fits 8 GB with a wide margin: more roads per point
change the number of `/table` requests (four times as many), not the memory per request. The time varies with machine
load: a first run of this notebook measured 70 s and 192 s. A later end-to-end run (2026-10-29, #15, load 3 to 23) measured 42 s and 132 s (3.2 times),
`osrm-routed` 416 MB and the notebook process 566 MB; the pair counts and the slice results were unchanged. The one-snap table here has 1,251 more pairs than the #13
build (1,136,444); #13 snapped candidate sites only and let OSRM snap the hex centres inside `/table`, while
`snaps=1` here snaps both ends the same way as `snaps=2`, so it is the like-for-like before.

**Second road.** 1,410 of 1,458 candidate sites (96.7%) and 3,264 of 4,859 hex centres (67.2%) got a snap point
different from the nearest road's. The rest repeat the nearest road: no second road within 100 m lying more than 5 m
across it (open land at the area's edge, a lane with nothing parallel). Probably sites sit in the built-up core, where roads run parallel, and hex centres include the sparse outskirts (not checked).

**Slice** (same slice as 0.5: 299,666 pairs before, 308,100 after; nothing in the before table dropped out):

| | |
|---|---:|
| pairs that moved (more than 0.01 min) | 225,317 of 299,666 (75.2%) |
| faster by more than 0.5 min / more than 1 min | 64,013 (21.4%) / 27,985 (9.3%) |
| minutes saved over moved pairs: median / 90th percentile / max | 0.25 / 1.18 / 11.16 |
| pairs slower | 0 |
| newly within 17 min | 8,434 |
| newly within 5.0 min | 3,711 |

Most pairs moved, but by little: the median saving is 15 s, because the fastest of four takes a few seconds off
almost any pair by choosing a better entry onto the road, not only by avoiding a far carriageway. The tail is the fix:
a tenth of the pairs gain over a minute and the largest gain is 11 min, and 3,711 pairs enter the default scenario's
5.0-minute catchments (1.2% of the slice's pairs; over the whole area the default-scenario pairs grow by 9,693). Nothing got slower, as it must when
the minimum includes the old time.

**Out-and-back on 200 short slice pairs** (before time at most 6 min, seed 42; "before" route time equals the table's
exactly, max difference 0.000 min):

| | before | after |
|---|---:|---:|
| more than 300 m at either end | 29 (14.5%) | 19 (9.5%) |
| start end (candidate site) | 14 | 6 |
| hex end | 16 | 13 |
| median start/end path | 114 m | 108 m |

(An end count adds to more than the pair count because a pair can loop at both ends.) The share drops from 14.5% to
9.5%, mainly at the candidate end (14 to 6), where the two snaps were chosen to give a divided road's two carriageways;
0.5 measured 15% (6 of 40) on a smaller sample. It is not gone. **19 pairs still ride more than 300 m at an end, and 16
of them are faster than before**: taking a better road did not remove the detour. Likely reasons, checked in the next section (#20): a
one-way or a median with no opening, where both carriageways of a divided road lead the same way out and back; a second
road that is none (the nearest road repeats) or is a service road; and the hex end, where 3,264 of 4,859 centres have a
distinct second road and the rest, 1,595 hexes, only the nearest one. Whether 9.5% matters for who is reachable is for the spot-check; the
snapping row of the assumptions table already says it is a fix by geometry, not by the road's name.

## Residual out-and-back (#20)

The 19 pairs above that still ride more than 300 m at an end: why, and would another second-road rule fix them? Two
steps on the same 200 pairs. First, each looped end is sorted into one cause, using the OSM tags of its two roads
(`osmium getparents` on the extract gives the ways through each snap point's nodes). Second, two variants of the
second-road rule, alone and together, are run and counted the same way.

Causes, in this order:

- **no second road**: the end's second road is the nearest road repeated.
- **second road not the far carriageway**: the nearest road is one-way, and the second one is not its far carriageway.
  Far means one-way as well, same highway class, and the same name unless one of the two has none. One-way is
  the explicit `oneway=yes` tag; none of the 19 ends' roads is a roundabout or `oneway=-1`, the implicit cases.
- **both carriageways loop**: the second road is the far carriageway, and the route from it (the other end held at its
  chosen snap) also rides more than 300 m.
- **something else**: the rest.

Variants: **same name preferred** (hex end: take the nearest later hit with the first road's name, if one qualifies),
**hex centres within 200 m** (the second road may lie up to 200 m away; 60 `/nearest` hits so that it is found) and both.
The candidate end keeps the rule as built.

In [ ]:
import itertools

import requests

from catchment.reach import NEAREST_SEGMENTS, nearest_roads

def nearest_hits(url: str, df: pd.DataFrame, keys: Iterable[str], number: int) -> dict[str, tuple[float, float, list[dict[str, Any]]]]:
    """/nearest hits of each key's centre: {key: (x, y, waypoints)}."""
    out = {}
    for k in keys:
        x, y = df.loc[k, "lon"], df.loc[k, "lat"]
        r = requests.get(f"{url}/nearest/v1/driving/{x:.6f},{y:.6f}", params={"number": number}, timeout=300).json()
        out[k] = (x, y, r["waypoints"])
    return out

def pt(w: dict[str, Any]) -> tuple[float, float]:
    return tuple(w["location"])

routes = {}   # (src, dst) -> route_pair result; many pairs share a snap

def evaluate(url: str, c_roads: dict[str, list[dict[str, Any]]], h_roads: dict[str, list[dict[str, Any]]]) -> pd.DataFrame:
    """One row per sample pair: the fastest of the four routes between the ends' two roads (an end with one road repeats
    it): minutes, leave and arrive m, and which snap each end used (0 nearest, 1 second)."""
    rows = []
    for p in samples.itertuples():
        cr, hr = c_roads[p.candidate], h_roads[p.hex]
        cr, hr = cr + cr[:1] * (2 - len(cr)), hr + hr[:1] * (2 - len(hr))
        for c in cr:
            for h in hr:
                if (pt(c), pt(h)) not in routes:
                    routes[pt(c), pt(h)] = route_pair(url, pt(c), pt(h))
        a, b = min(itertools.product((0, 1), (0, 1)), key=lambda ab: routes[pt(cr[ab[0]]), pt(hr[ab[1]])][0])
        mins, leave, arrive = routes[pt(cr[a]), pt(hr[b])]
        rows.append({"candidate": p.candidate, "hex": p.hex, "mins": mins, "leave": leave, "arrive": arrive, "a": a, "b": b,
                     "cr": cr, "hr": hr, "c_distinct": pt(cr[1]) != pt(cr[0]), "h_distinct": pt(hr[1]) != pt(hr[0])})
    return pd.DataFrame(rows)

with served_osrm(DATA) as router:
    c_hits = nearest_hits(router.url, cand_cmd, set(samples.candidate), NEAREST_SEGMENTS)
    h_hits = {n: nearest_hits(router.url, area_hexes, set(samples.hex), n) for n in (NEAREST_SEGMENTS, 60)}
    built = lambda hits: {k: nearest_roads(w, x, y, 2) for k, (x, y, w) in hits.items()}
    local = lambda hits, max_m, same_name: {k: nearest_roads(w, x, y, 2, max_m=max_m, same_name=same_name) for k, (x, y, w) in hits.items()}
    c_built = built(c_hits)
    # area-wide: hex centres with no distinct second road within 100 m, and how many find one within 200 m
    all_hits = nearest_hits(router.url, area_hexes, area_hexes.index, 60)
    lone = [k for k, (x, y, w) in all_hits.items() if len(nearest_roads(w, x, y, 2)) < 2]
    gain = sum(len(nearest_roads(all_hits[k][2], *all_hits[k][:2], 2, max_m=200)) == 2 for k in lone)
    variants = {   # name -> hex end's second roads; the candidate end is always as built
        "as built": built(h_hits[NEAREST_SEGMENTS]),
        "same name preferred": local(h_hits[NEAREST_SEGMENTS], assumptions.SECOND_ROAD_MAX_M, True),
        "hex centres within 200 m": local(h_hits[60], 200, False),
        "both": local(h_hits[60], 200, True),
    }
    runs = {name: evaluate(router.url, c_built, h_roads) for name, h_roads in variants.items()}

# the section must reproduce the "after" of the previous section
base = runs["as built"]
assert (base[["leave", "arrive"]].max(axis=1).to_numpy() > 300).tolist() == (samples["after path m"] > 300).tolist(), "differs from the after above"
assert np.allclose(base.mins, samples["after min"]), "differs from the after above"
print("as built reproduces the after:", int((samples["after path m"] > 300).sum()), "pairs > 300 m;", len(routes), "routes")
print(f"area hexes with no second road within 100 m (60 hits): {len(lone):,} of {len(area_hexes):,}; "
      f"of them, {gain:,} find one within 200 m")

In [ ]:
import subprocess

# OSM tags of every looped end's two roads: OSRM's waypoint nodes are OSM node ids ([x, x] on a one-way segment)
looped = base[base[["leave", "arrive"]].max(axis=1) > 300]
node_ids = sorted({n for r in looped.itertuples() for w in (*r.cr, *r.hr) for n in w["nodes"] if n})
with tempfile.TemporaryDirectory() as tmp:
    ids_file = Path(tmp) / "ids.txt"
    ids_file.write_text("\n".join(f"n{n}" for n in node_ids))
    opl = subprocess.run(["osmium", "getparents", str(DATA / "osm.osm.pbf"), "-i", str(ids_file), "-f", "opl", "-o", "-", "--overwrite"],
                         capture_output=True, text=True, check=True).stdout
ways_of = {}   # node id -> [(highway, oneway)] of the ways through it
for line in opl.splitlines():
    if line.startswith("w"):
        f = {t[0]: t[1:] for t in line.split(" ") if t}
        tags = dict(t.split("=", 1) for t in f.get("T", "").split(",") if "=" in t)
        for n in f.get("N", "").split(","):
            ways_of.setdefault(int(n[1:]), []).append((tags.get("highway"), tags.get("oneway") == "yes"))

def road_tags(w: dict[str, Any]) -> tuple[str | None, bool]:
    """(highway, one-way) of the way through both of a hit's nodes (the one node if both are the same)."""
    a, b = w["nodes"]
    both = set(ways_of.get(a, [])) if a == b else set(ways_of.get(a, [])) & set(ways_of.get(b, []))
    return sorted(both, key=str)[0] if both else (None, False)

def describe(w: dict[str, Any]) -> str:
    hw, oneway = road_tags(w)
    return f"{hw}{' [1-way]' if oneway else ''} {w['name']} @{w['distance']:.0f} m"

rows = []
for i, r in looped.iterrows():
    for end, loop_m, roads, chosen, other, other_chosen in (("candidate", r.leave, r.cr, r.a, r.hr, r.b),
                                                           ("hex", r.arrive, r.hr, r.b, r.cr, r.a)):
        if loop_m <= 300:
            continue
        first, second = roads
        (hw0, one0), (hw1, one1) = road_tags(first), road_tags(second)
        far = one0 and one1 and hw0 == hw1 and (first["name"] == second["name"] or not (first["name"] and second["name"]))
        alt = roads[1 - chosen]   # the other snap at this end, the other end held at its chosen snap
        alt_route = routes[pt(alt), pt(other[other_chosen])] if end == "candidate" else routes[pt(other[other_chosen]), pt(alt)]
        alt_loop = alt_route[1] if end == "candidate" else alt_route[2]
        if pt(second) == pt(first):
            cause = "no second road"
        elif one0 and not far:
            cause = "second road not the far carriageway"
        elif far and alt_loop > 300:
            cause = "both carriageways loop"
        else:
            cause = "something else"
        rows.append({"pair": i, "end": end, "path m": round(loop_m), "first road": describe(first), "second road": describe(second),
                     "other snap's loop m": round(alt_loop), "cause": cause})
causes = pd.DataFrame(rows)
order = ["no second road", "second road not the far carriageway", "both carriageways loop", "something else"]
counts = causes.groupby(["cause", "end"]).size().unstack(fill_value=0).reindex(order, fill_value=0)
counts["total"] = counts.sum(axis=1)
display(causes.sort_values(["cause", "pair"], key=lambda s: s.map(order.index) if s.name == "cause" else s)
        .style.hide(axis="index").set_properties(**{"text-align": "left"}))
display(counts)
print(f"looped ends: {len(causes)} on {len(looped)} pairs; pairs looping at both ends: {int((looped.leave > 300).mul(looped.arrive > 300).sum())}")

In [ ]:
def variant_row(df: pd.DataFrame) -> dict[str, int | float | str]:
    loop = df[["leave", "arrive"]].max(axis=1) > 300
    return {"pairs > 300 m at either end": int(loop.sum()), "share": f"{loop.mean():.1%}",
            "candidate end": int((df.leave > 300).sum()), "hex end": int((df.arrive > 300).sum()),
            "median min": round(df.mins.median(), 3),
            "sample hexes with a distinct second road": f"{df.drop_duplicates('hex').h_distinct.mean():.1%}",
            "pairs whose route changed vs as built": int((~np.isclose(df[["mins", "leave", "arrive"]], base[["mins", "leave", "arrive"]])).any(axis=1).sum())}

display(pd.DataFrame({name: variant_row(df) for name, df in runs.items()}).T)

Results from the run on 2026-10-29 (copied here because outputs are stripped on commit; re-running updates the printed
tables, not this one). Each of the 19 pairs loops at exactly one end, so there are 19 looped ends.

**Cause of each looped end** (end: candidate site / hex):

| cause | candidate | hex | total |
|---|---:|---:|---:|
| no second road | 0 | 0 | 0 |
| second road not the far carriageway | 1 | 3 | 4 |
| both carriageways loop | 1 | 3 | 4 |
| something else | 4 | 7 | 11 |

**Variants of the second-road rule** (200 pairs; the candidate end is as built in all four):

| rule | pairs > 300 m at either end | candidate end | hex end | median min | sample hexes with a distinct second road | pairs whose route changed |
|---|---:|---:|---:|---:|---:|---:|
| as built | 19 (9.5%) | 6 | 13 | 4.372 | 98.2% | |
| same name preferred | 18 (9.0%) | 6 | 12 | 4.355 | 98.2% | 1 |
| hex centres within 200 m (60 hits) | 19 (9.5%) | 6 | 13 | 4.372 | 100% | 0 |
| both | 18 (9.0%) | 6 | 12 | 4.322 | 100% | 5 |

No pair lacks a second road: every looped end has one, so a missing second road is not the cause. The largest group,
11 ends, is the one the rule cannot fix. In 10 of them the nearest road is a two-way residential or service lane (the
11th is a two-way tertiary road beside a one-way secondary one). In 9 the other snap loops too; in the other 2 (pairs 75 and 120) the other snap loops
less but is slower, so the fastest of four keeps the one that loops. This is the lane network round a block: a route
from one lane to the next goes round the block. No rule about which road is second changes that. The 4 ends where the
second road is not the far carriageway have a one-way first road and a second of another class (a residential road, a
slip road, an underpass); the 4 where both carriageways loop are pairs of one-way residential roads.

Same name preferred fixes one pair: on Mehrauli-Gurgaon Road the rule took a one-way tertiary slip road at 8 m over the
far carriageway of the same road at 9 m. Raising the reach to 200 m for hex centres changes nothing here: 98.2% of the
sample's hexes already have a second road within 100 m (100% within 200 m), and no pair's route changes. Over the whole area, 1,595 of 4,859 hex centres have no second road within 100 m, and 781
of them find one within 200 m. Those hexes are probably on the outskirts (not checked), outside the slice sample, so
this sample cannot say what the longer reach does there.

The rule stays as built (decision log, 2026-10-29). Whether 9.5% matters for who is reachable is for the spot check
(#16-#18).

## Area demand map

Demand per res-9 hex across the whole area, in orders a day, at the default capture share (read from the demand
table's attributes). Every hex of `demand-r9` is drawn; the outline is the area. The look is `00-demand`'s: magma,
white edges, a colorbar.


In [ ]:
capture = demand.attrs["capture_share"]
hexes = gpd.GeoDataFrame(demand[["demand"]],
                         geometry=[Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in demand.index], crs=4326)
fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)
hexes.plot(column="demand", ax=ax, cmap="magma", linewidth=0.1, edgecolor="white", legend=True,
           legend_kwds={"label": "orders per day per hex", "shrink": 0.8})
gpd.GeoSeries([area], crs=4326).boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_aspect(1 / np.cos(np.radians(28.44)))
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title(f"Demand at default capture share {capture:.3f}, whole area ({len(hexes):,} hexes, H3 res 9)")
fig.savefig(FIGS / "03-demand.png", dpi=120)
plt.show()
print(f"hexes {len(demand):,}; total demand {demand.demand.sum():,.0f} orders/day; capture share {capture:.4f}; "
      f"per hex: median {demand.demand.median():.1f}, max {demand.demand.max():.1f}; res {demand.attrs['res']}")


![Demand per hex across the area](figures/03-demand.png)


Results from the run on 2026-10-29 (copied here because outputs are stripped on commit): **4,859 hexes at H3 res 9**,
total demand **89,728 orders a day** at the default capture share of **0.0805** (about 8% of households order on a
given day). Per hex the median is 11.2 orders a day and the maximum 110.3. These totals are at res 9. The res-8 demand
table is built natively and is not comparable: it has 697 hexes and a total of 88,472 (1.4% below res 9's 89,728) orders a day, because it drops
the hexes whose centre is outside the area (see the "Demand and reach (stage 1) design" entry in the decision log).

The brightest hexes are in the north-east (the South Delhi end of the area); the middle band is moderate, and the
west and south edges are dark.


## Full-area solve time

The default scenario has to return a plan within 10 s, with its proven gap to the bound stated (the planner's
definition, 2026-10-30). The 0.5 model, measured first below, misses that on the placeholder network; the tighter model
below meets it. 0.5 extrapolated the solve from the slice (3.1-5.6 s for
N = 10 on the full area; three slice copies at the area's pair count solved in 1.2 s). Here the 0.5 CP-SAT model runs on
the real full-area tables: all of `demand-r9`'s hexes, the candidate sites in rank order, and the pairs of the table
reachable at the default scenario. The model is copied inline from `02-plan` (stage 2 moves it into the package),
changed only where the case is built: it reads the area's tables, not the slice's, and the constants come from
`assumptions.py`. The placeholder network is placed over the area's candidate sites by the same rule as on the slice.

**Main run: the placeholder network.** 3 operator, 4 Competitor A and 3 Competitor B stores (10, sized for the slice
and kept as it is: a deliberately thin network for an area that holds about 57 stores' worth of orders), N = 10 new
stores, default capacity. The time is `solve_plan` as a whole, model build plus solve. **Sensitivity run:** a network
sized to the area, round(area demand ÷ orders per store per day) stores, with each brand's share of `BRAND_COUNTS` (3:4:3) kept. More existing stores
means fewer open sites and, through competitor reach, different shares. Both run at X = 30 (1,458 candidate sites).

The plan rows of the assumptions table follow. The rows that say "slice" (brands and store counts, store placement, sites
for new stores) hold here for the area's candidate sites; the counts are the slice's placeholder for the main run.


In [ ]:
rows = assumptions.table()
plan_rows = ["Default scenario", "New stores, N", "Orders per store per day", "Capacity per store", "Brands and store counts",
             "Store placement", "Competitor reach", "Demand split", "Sites for new stores", "Assignment tie-break",
             "Integer scale", "Solver"]
pd.set_option("display.max_colwidth", None)
display(rows[rows.Input.isin(plan_rows)].style.hide(axis="index").set_properties(**{"text-align": "left"}))


In [ ]:
CAP_INT = round(CAPACITY * SCALE)

def reachable(ff_min: float | np.ndarray | pd.Series) -> bool | np.ndarray | pd.Series:
    """Hex reachable from a site: free-flow minutes x multiplier + pick-and-pack <= promise."""
    return ff_min * MULTIPLIER + PICK_AND_PACK <= PROMISE + 1e-9

@dataclass
class Case:
    res: int
    hexes: pd.Index
    demand: np.ndarray        # orders per day per hex
    sites: pd.Index           # candidate sites, in rank order
    rank: np.ndarray          # places-count rank in the whole area, 0 = busiest
    ps: np.ndarray            # reachable pairs: site position
    ph: np.ndarray            # reachable pairs: hex position
    pm: np.ndarray            # reachable pairs: free-flow minutes
    n_area_hexes: int         # hexes in the whole area
    stores: dict[str, list[int]]   # brand -> site positions
    k: np.ndarray             # competitor brands reaching each hex
    share_int: np.ndarray     # operator share of each hex if it serves it, hundredths of an order
    new_sites: np.ndarray     # site positions open for a new store
    cap_int: int              # capacity, hundredths of an order

def load_case(res: int, hexes: pd.DataFrame, candidates: pd.DataFrame, table: pd.DataFrame) -> Case:
    """The full area at one resolution: every hex of the demand table, every candidate site in rank order, and the
    table's pairs reachable at the default scenario."""
    cand = candidates.sort_values("rank")
    tab = table[reachable(table.minutes)]
    spos = pd.Series(np.arange(len(cand)), index=cand.index)
    hpos = pd.Series(np.arange(len(hexes)), index=hexes.index)
    d = hexes["demand"].to_numpy()
    return Case(res, hexes.index, d, cand.index, cand["rank"].to_numpy(), spos[tab.candidate].to_numpy(),
                hpos[tab.hex].to_numpy(), tab.minutes.to_numpy(), len(hexes), {}, np.zeros(len(d), int),
                np.round(d * SCALE).astype(np.int64), np.arange(len(cand)), CAP_INT)

def describe(case: Case) -> None:
    per_hex = np.bincount(case.ph, minlength=len(case.hexes))
    print(f"res {case.res}: hexes {len(case.hexes):,}; candidate sites {len(case.sites):,}; default-scenario pairs {len(case.ps):,}")
    print(f"hexes no candidate reaches: {(per_hex == 0).sum():,}, demand {case.demand[per_hex == 0].sum():,.0f} of {case.demand.sum():,.0f} orders/day")
    print(f"largest catchment's demand {np.bincount(case.ps, weights=case.demand[case.ph]).max():,.0f} orders/day against capacity {CAPACITY:,.0f}")

area_9 = load_case(9, demand, cand_cmd, table_cmd)
describe(area_9)
print(f"table: {len(table_cmd):,} pairs within {assumptions.MAX_PAIR_MIN} min")


In [ ]:
def place_network(C: Case, counts: dict[str, int] = BRAND_COUNTS, seed: int = SEED) -> dict[str, list[int]]:
    rng = np.random.default_rng(seed)
    reached = np.bincount(C.ps, weights=C.demand[C.ph], minlength=len(C.sites))
    chosen = rng.choice(len(C.sites), size=sum(counts.values()), replace=False, p=reached / reached.sum())
    labels = rng.permutation(np.repeat(list(counts), list(counts.values())))
    return {b: sorted(chosen[labels == b].tolist()) for b in counts}

def with_network(C: Case, stores: dict[str, list[int]], cap_int: int | None = None) -> Case:
    comp = [b for b in stores if b != "Operator"]
    hit = np.zeros((len(comp), len(C.hexes)), bool)
    for i, b in enumerate(comp):
        hit[i, C.ph[np.isin(C.ps, stores[b])]] = True
    k = hit.sum(axis=0)
    occupied = np.array(sorted(s for ss in stores.values() for s in ss))
    assert len(occupied) == len(set(occupied)), "two stores on one site"
    return replace(C, stores=stores, k=k, share_int=np.round(C.demand / (1 + k) * SCALE).astype(np.int64),
                   new_sites=np.setdiff1d(np.arange(len(C.sites)), occupied), cap_int=cap_int or C.cap_int)

def solve_plan(C: Case, N: int, new_sites: np.ndarray | None = None, pair_model: bool = False,
               time_limit: float = TIME_LIMIT) -> dict[str, str | float | list[int]]:
    """Choose N new sites from new_sites (default: all open ones) to maximise the operator's orders.
    pair_model=True builds the plain model (x for every pair); otherwise only sites with W_s > capacity get x."""
    t0 = time.perf_counter()
    new = C.new_sites if new_sites is None else new_sites
    op = np.array(C.stores["Operator"])
    is_new, is_op = np.zeros(len(C.sites), bool), np.zeros(len(C.sites), bool)
    is_new[new], is_op[op] = True, True
    keep = (is_new | is_op)[C.ps]
    ps, ph = C.ps[keep], C.ph[keep]
    reach_w = np.bincount(ps, weights=C.share_int[ph], minlength=len(C.sites))      # W_s
    constrained = np.ones(len(C.sites), bool) if pair_model else reach_w > C.cap_int
    model = cp_model.CpModel()
    y = {int(s): model.NewBoolVar(f"y{s}") for s in new}
    model.Add(sum(y.values()) == N)
    terms, coeffs, const = [], [], 0
    x_at = defaultdict(list)
    for h, sites_h in pd.Series(ps).groupby(ph):
        sites_h = sites_h.to_numpy()
        w = int(C.share_int[h])
        free = sites_h[~constrained[sites_h]]
        if is_op[free].any():               # an existing store that cannot hit capacity covers this hex
            const += w
            continue
        row = []
        if len(free):
            covered = model.NewBoolVar("")
            model.AddBoolOr([y[int(s)] for s in free]).OnlyEnforceIf(covered)
            row.append(covered)
        for s in sites_h[constrained[sites_h]]:
            xv = model.NewBoolVar("")
            if is_new[s]:
                model.AddImplication(xv, y[int(s)])
            x_at[int(s)].append((xv, w))
            row.append(xv)
        if len(row) > 1:
            model.AddAtMostOne(row)
        terms += row
        coeffs += [w] * len(row)
    for lst in x_at.values():
        model.Add(cp_model.LinearExpr.WeightedSum([v for v, _ in lst], [w for _, w in lst]) <= C.cap_int)
    model.Maximize(cp_model.LinearExpr.WeightedSum(terms, coeffs) + const)
    t1 = time.perf_counter()
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.num_workers = WORKERS
    solver.parameters.random_seed = SOLVER_SEED
    status = solver.Solve(model)
    t2 = time.perf_counter()
    ok = status in (cp_model.OPTIMAL, cp_model.FEASIBLE)
    return {"status": solver.StatusName(status), "chosen": sorted(s for s in y if solver.Value(y[s])) if ok else [],
            "obj": solver.ObjectiveValue() / SCALE, "bound": solver.BestObjectiveBound() / SCALE,
            "pairs": len(ps), "new_sites": len(new), "hexes": int(len(np.unique(ph))),
            "constrained sites": int(constrained[np.unique(ps)].sum()),
            "build_s": t1 - t0, "solve_s": t2 - t1, "total_s": t2 - t0}

def evaluate(C: Case, open_sites: list[int], time_limit: float = 30) -> tuple[np.ndarray, int]:
    """Best assignment of hexes to the operator's open stores within capacity. Returns (owner per hex, orders x 100)."""
    keep = np.isin(C.ps, list(open_sites))
    ps, ph, pm = C.ps[keep], C.ph[keep], C.pm[keep]
    w = C.share_int[ph]
    model = cp_model.CpModel()
    x = [model.NewBoolVar("") for _ in range(len(ps))]
    for _, js in pd.Series(np.arange(len(ps))).groupby(ph):
        if len(js) > 1:
            model.AddAtMostOne([x[j] for j in js])
    for _, js in pd.Series(np.arange(len(ps))).groupby(ps):
        js = js.to_numpy()
        model.Add(cp_model.LinearExpr.WeightedSum([x[j] for j in js], [int(w[j]) for j in js]) <= C.cap_int)
    served = cp_model.LinearExpr.WeightedSum(x, [int(v) for v in w])
    model.Maximize(served)                      # stage 1: orders served; always solved to optimality
    solver = cp_model.CpSolver()
    solver.parameters.num_workers = 1
    solver.parameters.random_seed = SOLVER_SEED
    solver.parameters.max_time_in_seconds = time_limit
    status = solver.Solve(model)
    assert status == cp_model.OPTIMAL, solver.StatusName(status)
    best = round(solver.ObjectiveValue())
    for v in x:
        model.AddHint(v, solver.Value(v))
    model.Add(served == best)                   # stage 2: the same orders, the least total travel time (hundredths of a minute)
    model.Minimize(cp_model.LinearExpr.WeightedSum(x, [round(float(m) * 100) for m in pm]))
    solver.parameters.max_time_in_seconds = 5   # best effort: stage 1 fixes the orders, this only picks among ties
    status = solver.Solve(model)
    assert status in (cp_model.OPTIMAL, cp_model.FEASIBLE), solver.StatusName(status)
    owner = np.full(len(C.hexes), -1)
    for j in np.flatnonzero([solver.Value(v) for v in x]):
        owner[ph[j]] = ps[j]
    assert int(C.share_int[owner >= 0].sum()) == best
    return owner, best


### The runs

Each run builds the case (`place_network`, `with_network`), then times `solve_plan` three times, build and solve
together. After the timed runs the CP-SAT objective is checked against `evaluate` for the chosen stores, as in 02-plan.
The 60 s time limit and 8 workers are the assumptions table's.


In [ ]:
def timed_runs(C: Case, N: int, new_sites: np.ndarray | None = None, repeats: int = 3,
               solver: Callable[..., dict[str, str | float | list[int]]] = solve_plan) -> list[dict[str, str | float | list[int]]]:
    runs = []
    for _ in range(repeats):
        load = os.getloadavg()[0]
        r = solver(C, N, new_sites=new_sites)
        r["load_before"] = load
        runs.append(r)
    return runs

def summary(name: str, C: Case, runs: list[dict[str, str | float | list[int]]], check: str = "optimal") -> dict[str, str | float]:
    """check: 'always' runs evaluate even if CP-SAT is not proven optimal (to time it); 'optimal' only if proven; 'never'."""
    r = runs[0]
    t = np.array([x["total_s"] for x in runs])
    for i, x in enumerate(runs):
        print(f"{name}, run {i + 1}: {x['status']}, objective {x['obj']:.2f}, bound {x['bound']:.2f}, build {x['build_s']:.2f} s, "
              f"solve {x['solve_s']:.2f} s, total {x['total_s']:.2f} s, load before {x['load_before']:.1f}")
    same = all(x["status"] == r["status"] and abs(x["obj"] - r["obj"]) < 0.005 for x in runs)   # alternative optima may pick different sites
    # the CP-SAT objective must equal what the evaluator serves for the chosen stores
    ev, ev_s = float("nan"), float("nan")
    if check == "always" or (check == "optimal" and r["status"] == "OPTIMAL"):
        ev_s = time.perf_counter()
        try:
            _, served_int = evaluate(C, list(C.stores["Operator"]) + r["chosen"])
            ev = served_int / SCALE
            if r["status"] == "OPTIMAL":
                assert abs(r["obj"] - ev) < 0.005, (r["status"], r["obj"], ev)
        except AssertionError as e:
            if r["status"] == "OPTIMAL" and "FEASIBLE" not in str(e):
                raise
            # evaluate's own stage 1 did not reach OPTIMAL within its 30 s limit
        ev_s = time.perf_counter() - ev_s
    print(f"{name}: evaluate serves {ev:.2f} for run 1's stores (CP-SAT objective {r['obj']:.2f}), {ev_s:.1f} s; runs agree: {same}")
    catch = np.bincount(C.ps, weights=C.demand[C.ph], minlength=len(C.sites))
    return {"network": name, "existing stores (op/A/B)": "/".join(str(len(C.stores[b])) for b in C.stores),
            "status": r["status"], "objective (orders/day)": round(r["obj"], 2), "bound": round(r["bound"], 2),
            "pairs": r["pairs"], "hexes": r["hexes"], "new sites": r["new_sites"], "constrained sites": r["constrained sites"],
            "largest catchment (orders/day)": round(catch.max()), "capacity": round(CAPACITY),
            "build s (median)": round(float(np.median([x["build_s"] for x in runs])), 2),
            "solve s (median)": round(float(np.median([x["solve_s"] for x in runs])), 2),
            "total s median": round(float(np.median(t)), 2), "min": round(float(t.min()), 2), "max": round(float(t.max()), 2),
            "all totals": ", ".join(f"{v:.2f}" for v in t),
            "gap % (median)": round(float(np.median([100 * (x["bound"] - x["obj"]) / x["obj"] if x["obj"] > 0 else np.nan for x in runs])), 2),   # nan: no plan found
            "load before": ", ".join(f"{x['load_before']:.1f}" for x in runs), "evaluate s": round(ev_s, 1), "evaluate orders/day": round(ev, 2)}

print(f"OR-tools {ortools.__version__}; os.cpu_count() {os.cpu_count()}; load average at start (1, 5, 15 min): "
      + ", ".join(f"{v:.2f}" for v in os.getloadavg()))

# main run: the placeholder network (3/4/3), N = 10
placeholder_9 = with_network(area_9, place_network(area_9))
runs_main = timed_runs(placeholder_9, N_AREA)
res_main = summary("X=30 placeholder 3/4/3", placeholder_9, runs_main, check="always")

# sensitivity run: a network sized to the area, split 3:4:3 in proportion
n_net = round(area_9.demand.sum() / ORDERS_PER_STORE_DAY)
counts_sized = {b: round(n_net * n / sum(BRAND_COUNTS.values())) for b, n in BRAND_COUNTS.items()}   # BRAND_COUNTS' proportions
assert sum(counts_sized.values()) == n_net
sized_9 = with_network(area_9, place_network(area_9, counts_sized))
runs_sized = timed_runs(sized_9, N_AREA)
res_sized = summary(f"X=30 sized to the area {'/'.join(map(str, counts_sized.values()))}", sized_9, runs_sized)

display(pd.DataFrame([res_main, res_sized]).set_index("network").T)
print("load average (1 min) before each timed run, main:", [round(x["load_before"], 2) for x in runs_main],
      "sensitivity:", [round(x["load_before"], 2) for x in runs_sized])
print("load average at end (1, 5, 15 min): " + ", ".join(f"{v:.2f}" for v in os.getloadavg()))
print("area demand", f"{area_9.demand.sum():,.0f}", "÷", f"{ORDERS_PER_STORE_DAY:,.1f}", "=", f"{area_9.demand.sum() / ORDERS_PER_STORE_DAY:.2f}", "->", n_net, counts_sized)


Results from the end-to-end run of this notebook on 2026-10-29 (copied here because outputs are stripped on commit;
re-running updates the printed tables, not these). Apple M1, 8 cores, 8 GB RAM, OR-tools 9.15.6755, 8 workers, 60 s
limit, solver seed 1. **The load matters, and it was not constant.** The 1-minute load average on the 8 cores was 3 to
19 before the three main runs (3.3, 14.3, 19.1), 11 to 13 before the area-sized runs, 13 to 23 before the X cuts, 18
before the diagnostic and 16 to 18 before the res-8 runs, from other jobs on the machine. Earlier runs of the same cells
at loads of 17 to 40 gave different times: the area-sized network took 14 to 42 s (median 14.3, 21.0 and 34.8 s in
three runs), and res 8 stopped at the 60 s limit. Nothing here was measured on an idle machine.

The case: 4,859 hexes, 1,458 candidate sites, 59,855 default-scenario pairs (free-flow ≤ 5.0 min; 1,187,469 in the
table). 1,557 hexes with 5,839 of the area's 89,728 orders a day are reached by no candidate (6.5%), so coverage at
X = 30 is 93.5%. The largest catchment holds **5,266 orders a day against a capacity of 2,364**.

| X = 30, N = 10 | main: placeholder 3/4/3 | sensitivity: sized to the area, 17/23/17 (57 stores) |
|---|---:|---:|
| status | FEASIBLE (stopped at the 60 s limit) | OPTIMAL |
| total time, median (min / max) | **60.3 s** (60.3 / 60.6) | **6.1 s** (5.8 / 6.4) |
| build / solve, median | 0.2 s / 60.1 s | 0.1 s / 6.0 s |
| load before the runs | 3.3, 14.3, 19.1 | 11.3, 11.4, 12.7 |
| objective, orders a day | 28,266 (runs: 28,266, 28,267, 28,266) | 30,401.05 |
| bound | 33,432 (18.3% above the objective) | 30,401.05 |
| pairs, hexes, sites open for new stores | 59,532, 3,302, 1,448 | 57,944, 3,300, 1,401 |
| constrained sites (W_s > capacity) | **123** | 22 |
| `evaluate` for the chosen stores | not finished: its first solve hit its 30 s limit without a proof | 30,401.05, equal to the objective (0.4 s) |

Area demand ÷ orders per store per day = 89,728 ÷ 1,576.1 = 56.9, so the sensitivity network has 57 stores: each brand's
share of `BRAND_COUNTS` (3:4:3) of 57, rounded, is 17 operator, 23 Competitor A, 17 Competitor B. The same seed places
both networks.

**Two networks, two verdicts.** **With the placeholder network (the thin 3/4/3, sized for the slice) the main run misses
the bar by a factor of six or more**, and it is not close to proven: after 60 s the best plan is still 18.3% below the
bound. The first of its three runs started at a load of 3.3 and took as long as the others, so the load does not explain
this miss. **With a network sized to the area (57 stores) the same model meets the bar: 6.1 s, proven optimal**, at a load
of 11 to 13 (not an idle machine). It is load-dependent: the same cell took 14 to 42 s at loads of 17 to 40. So the bar
is met when the network is sized to the area and the machine is not heavily loaded, and missed with the placeholder
network at every setting tried.

**The cause is capacity binding on the area.** 0.5 measured the slice, where the largest catchment (1,981 orders a day)
was under the capacity (2,364), no site was constrained and the problem was maximum coverage, extrapolated to 3.1-5.6 s
(three slice copies: 1.2 s). On the area the dense north-east gives catchments of up to 5,266 orders a day, and sites
that exceed capacity need assignment variables and a knapsack constraint: 123 of them with the thin network, 22 with
the sized one (more existing stores take sites and shares away from the new stores). The diagnostic below confirms it:
the same network and sites with capacity effectively infinite solve to proven optimality in 0.8 s. This is the 0.5 open
point "capacity that binds is slow", which turns out to bite at the default scenario. For the same reason `evaluate`
(the scorer, which assigns hexes to a plan's stores from scratch) is slow when capacity binds: its first solve did not
reach OPTIMAL in 30 s for the main run's 13 stores, while it took 0.4 s for the sized network's stores and 0.3 s for
res 8's. That check was skipped for the X cuts (it is only made when CP-SAT proves optimality, and none did).


### Cutting X

If the default scenario misses 10 s, the 0.5 entry says to cut X before the resolution. X is the share of the area's hexes
that may hold a new store. The X% candidate set is the X = 30 set with `rank < cutoff`, `cutoff = n_top(X, n_hexes, n_with_places)`: a
hex's rank does not depend on X, the candidates are those under that cut-off, and a pair's minutes depend only on its two
endpoints' snaps, so the X% table is the X = 30 table filtered to those candidates. This is equivalent to
`catchment-build reach --x-pct X` and needs no OSRM run. `n_top` is the package's.

The network is placed once, at X = 30, and keeps its stores at every X: only the sites open to new stores shrink,
`new = new_sites[rank[new_sites] < cutoff]` as in the 02-plan grid. X goes down 20, 10, 5 and stops at the first X whose median
total is at most 10 s with status OPTIMAL. Coverage is the share of the area's demand on hexes that some candidate site
at that X reaches at the default scenario. The last row is a labelled diagnostic.


In [ ]:
n_hexes, n_with_places = len(demand), len(cand_cmd)   # all of demand-r9 lies in the area; every X=30 candidate has a place
assert n_top(30, n_hexes, n_with_places) == len(cand_cmd)   # and 1,458 exceeds n_top at every smaller X, so the min never binds

def coverage(case: Case, cutoff: int) -> float:
    """Share of the area's demand on hexes that a candidate site with rank < cutoff reaches at the default scenario."""
    hit = np.unique(case.ph[case.rank[case.ps] < cutoff])
    return case.demand[hit].sum() / case.demand.sum()

cov30 = coverage(area_9, len(area_9.sites))
res_x = [dict(res_main, X=30, candidates=len(area_9.sites), coverage=cov30)]
fits = None
for x_pct in (20, 10, 5):
    cutoff = n_top(x_pct, n_hexes, n_with_places)
    new = placeholder_9.new_sites[placeholder_9.rank[placeholder_9.new_sites] < cutoff]
    runs = timed_runs(placeholder_9, N_AREA, new_sites=new)
    row = dict(summary(f"X={x_pct}", placeholder_9, runs), X=x_pct, candidates=cutoff, coverage=coverage(area_9, cutoff))
    res_x.append(row)
    if row["status"] == "OPTIMAL" and row["total s median"] <= SOLVE_BAR_S:
        fits = x_pct
        break
print("X that fits:", fits)

# diagnostic, not a scenario: the same network and X = 30 with capacity effectively infinite (plain maximum coverage)
infinite_9 = with_network(area_9, placeholder_9.stores, cap_int=10**12)
res_inf = dict(summary("X=30 DIAGNOSTIC infinite capacity", infinite_9, timed_runs(infinite_9, N_AREA), check="never"),
               X="30, infinite capacity", candidates=len(area_9.sites), coverage=cov30)
cols = ["X", "candidates", "coverage", "status", "objective (orders/day)", "bound", "constrained sites", "new sites", "pairs",
        "build s (median)", "solve s (median)", "total s median", "min", "max", "all totals"]
display(pd.DataFrame(res_x + [res_inf])[cols].set_index("X").round(4))
print(f"coverage at X=30: {cov30:.4f}; " + ", ".join(f"X={r['X']}: {r['coverage']:.4f}" for r in res_x[1:]))
print("load average at end (1, 5, 15 min): " + ", ".join(f"{v:.2f}" for v in os.getloadavg()))


**Cutting X, then the diagnostic.** All rows are the main network (3/4/3, seed 2026, placed once at X = 30), N = 10,
default capacity, median of three runs. Coverage is the share of the area's demand reached by some candidate site at
the default scenario. The objective is run 1's.

| X | candidate sites | coverage | status | total time, median (min / max) | objective | bound | constrained sites |
|---|---:|---:|---|---:|---:|---:|---:|
| 30 | 1,458 | 93.5% | FEASIBLE | 60.3 s (60.3 / 60.6) | 28,266 | 33,432 | 123 |
| 20 | 972 | 88.3% | FEASIBLE | 60.2 s (60.2 / 60.5) | 28,266 | 33,133 | 91 |
| 10 | 486 | 79.8% | FEASIBLE | 60.2 s (60.2 / 60.3) | 28,267 | 32,314 | 53 |
| 5 | 243 | 71.2% | FEASIBLE | 60.1 s (60.1 / 60.2) | 27,915 | 30,578 | 30 |
| **diagnostic**: X = 30, capacity effectively infinite | 1,458 | 93.5% | OPTIMAL | 0.8 s (0.8 / 1.0) | 33,697 | 33,697 | 0 |

**No X fits the placeholder network.** Every cut runs to the 60 s limit with an unproven plan, down to X = 5 (243 sites,
30 constrained sites, a bound still 9.5% above the best plan), at a coverage cost of 5 points at X = 20, 14 at X = 10 and
22 at X = 5. Cutting sites shrinks the model but leaves the constrained sites that make it hard. The X% candidate sets
here are the X = 30 table filtered by rank, equivalent to `catchment-build reach --x-pct X` (a hex's rank does not
depend on X and a pair's minutes only on its two endpoints' snaps). The existing stores stay where they were placed at
X = 30 at every X; only new-store sites shrink.


### Res 8, the last resort

No X fits, so the last resort is the coarser grid. `catchment-build reach --res 8` (X = 30) gives 697 hexes and 210
candidate sites; the table here was written by `build_reach` and `build.py`'s `write_parquet` against a separate OSRM
container. The network is placed with the same rule (3/4/3, seed 2026, probability proportional to the demand a site
reaches at the default scenario) over the res-8 candidates; N = 10 and the capacity are unchanged.

**Caveat: res-8 and res-9 numbers are not comparable.** The res-8 demand table is built natively at res 8, which drops the
population of res-8 hexes whose centre is outside the area (941,794 households against 955,163 at res 9, about 1.4%
lower; see the "Demand and reach (stage 1) design" entry in the decision log). Its totals, coverage and objective rest on
a slightly smaller demand.


In [ ]:
demand_r8 = pd.read_parquet(DATA / "demand-r8.parquet")
cand_r8 = pd.read_parquet(DATA / "candidates-r8-x30.parquet")
table_r8 = pd.read_parquet(DATA / "table-r8-x30.parquet")
print(f"households: res 8 {demand_r8.households.sum():,.0f} against res 9 {demand.households.sum():,.0f} "
      f"({demand_r8.households.sum() / demand.households.sum() - 1:+.2%}); demand: res 8 {demand_r8.demand.sum():,.0f} against "
      f"res 9 {demand.demand.sum():,.0f} orders/day ({demand_r8.demand.sum() / demand.demand.sum() - 1:+.2%})")
area_8 = load_case(8, demand_r8, cand_r8, table_r8)
describe(area_8)
print(f"table: {len(table_r8):,} pairs within {assumptions.MAX_PAIR_MIN} min")
placeholder_8 = with_network(area_8, place_network(area_8))
runs_8 = timed_runs(placeholder_8, N_AREA)
res_8 = dict(summary("res 8 X=30 placeholder 3/4/3", placeholder_8, runs_8), X=30, candidates=len(area_8.sites),
             coverage=coverage(area_8, len(area_8.sites)))
display(pd.DataFrame([dict(res_x[0], res=9), dict(res_8, res=8)])[["res"] + cols].set_index("res").T)
print(f"res 8 coverage {res_8['coverage']:.4f} against res 9 {cov30:.4f}; verdict against {SOLVE_BAR_S} s: "
      f"{res_8['status']}, median {res_8['total s median']} s")
print("load average at end (1, 5, 15 min): " + ", ".join(f"{v:.2f}" for v in os.getloadavg()))


Results from the same run (the table: `candidates-r8-x30` 210 sites, `table-r8-x30` 24,700 pairs; the container was
`catchment-osrm-15` on port 5015, 29 s including the table build). **Res-8 numbers are not comparable with res 9's** (the
caveat above): its demand is 88,472 orders a day on 697 hexes, 1.4% below res 9's 89,728.

| X = 30, main network 3/4/3 | res 9 | res 8 |
|---|---:|---:|
| hexes, candidate sites | 4,859, 1,458 | 697, 210 |
| coverage of demand at the default scenario | 93.5% | 78.9% |
| default-scenario pairs, kept in the run | 59,855, 59,532 | 1,194, 1,140 |
| constrained sites (W_s > capacity) | 123 | 18 |
| largest catchment's demand vs capacity | 5,266 vs 2,364 | 4,852 vs 2,364 |
| status | FEASIBLE | **OPTIMAL** |
| total time, median (min / max) | 60.3 s (60.3 / 60.6) | **25.5 s** (24.6 / 27.8) |
| objective, bound | 28,266; 33,432 | 26,619.37; 26,619.37 |
| `evaluate` for the chosen stores | not finished (30 s limit) | 26,619.37, equal to the objective (0.3 s) |

**Res 8 misses the bar too, but it is proven, not limit-bound:** 25.5 s, OPTIMAL, 1,140 pairs, at a coverage 14.6 points
lower than res 9's. An earlier, more loaded run of this cell stopped at the 60 s limit (FEASIBLE, 10.1% from its bound),
so its time is load-dependent as well. Size is not what makes the problem slow: 1,140 pairs still take 25 s because
capacity binds at 18 sites (the 0.5 entry's 0.04 s for res 8 assumed none). Not tried: res 8 with X cut as well.

**Verdict for the 0.5 model.** It **misses 10 s with the placeholder network at every setting the ticket allows** (res 9 at
X = 30, 20, 10 and 5; res 8 at X = 30), and meets it, at 6.1 s, only with a network sized to the area at res 9 and
X = 30, at a moderate load. **The tighter model in the next section meets the bar under the planner's definition**
(a plan within 10 s, its proven gap stated): all default-scenario runs return in under 10 s, with the placeholder
network's plan 16.8% from its bound on this machine and the other two cases proven optimal. So res 9 and X = 30
stand. What is left for stage 2 is binding capacity below the default (the capacity-1,000 stress case gets no usable
plan in 9 s), the slow `evaluate`, and moving the model into the package. The infinite-capacity row shows the
coverage part alone is fast (0.8 s).


### A tighter model

**Decision (the planner, 2026-10-30): the 10 s bar means a plan is returned within 10 s, with its proven gap to the bound
stated.** Proven optimality is not required. The 0.5 model, as measured above, misses on the placeholder network: it
returns its best plan after 60 s with a bound 18% above it. The weak part is the bound. For a new site `s` the capacity
row `Σ_h share_h x_{s,h} ≤ cap` does not depend on whether the site is open, and the only link to `y_s` is `x_{s,h} ≤
y_s` per pair, so in the linear relaxation a barely open site can still carry a whole store's capacity of small `x`
values. Make the row `Σ_h share_h x_{s,h} ≤ cap · y_s` instead: a closed site serves nothing. The row is valid, since
at an integer `y_s` it is the old row (open) or `x = 0` (closed), so the optimum is unchanged; it only tightens the
relaxation. The operator's existing stores keep `≤ cap`, and `x ≤ y` stays. In a separate experiment (not repeated
here) this moved the root bound on the main network from 33,432 to about 28,292, within 0.1% of the best plans.

`solve_plan_tight` below is `solve_plan` copied with that one change, and a budget of `SOLVE_LIMIT_S` = 9 s for build and solve (CP-SAT gets what the build leaves) so
that build plus solve stays under the bar. The change, in the capacity loop:

```diff
-    for lst in x_at.values():
-        model.Add(cp_model.LinearExpr.WeightedSum([v for v, _ in lst], [w for _, w in lst]) <= C.cap_int)
+    for s, lst in x_at.items():
+        used = cp_model.LinearExpr.WeightedSum([v for v, _ in lst], [w for _, w in lst])
+        model.Add(used <= C.cap_int * y[s] if is_new[s] else used <= C.cap_int)
```

(`evaluate` also gains a `time_limit` argument, default 30 s as before, for the check below.) The gap is
(bound − objective) ÷ objective, from the solver's best bound.


In [ ]:
def solve_plan_tight(C: Case, N: int, new_sites: np.ndarray | None = None, pair_model: bool = False,
                    time_limit: float = SOLVE_LIMIT_S) -> dict[str, str | float | list[int]]:
    """solve_plan with the capacity row of a new site tied to its y. Choose N new sites from new_sites (default: all open ones) to maximise the operator's orders.
    pair_model=True builds the plain model (x for every pair); otherwise only sites with W_s > capacity get x."""
    t0 = time.perf_counter()
    new = C.new_sites if new_sites is None else new_sites
    op = np.array(C.stores["Operator"])
    is_new, is_op = np.zeros(len(C.sites), bool), np.zeros(len(C.sites), bool)
    is_new[new], is_op[op] = True, True
    keep = (is_new | is_op)[C.ps]
    ps, ph = C.ps[keep], C.ph[keep]
    reach_w = np.bincount(ps, weights=C.share_int[ph], minlength=len(C.sites))      # W_s
    constrained = np.ones(len(C.sites), bool) if pair_model else reach_w > C.cap_int
    model = cp_model.CpModel()
    y = {int(s): model.NewBoolVar(f"y{s}") for s in new}
    model.Add(sum(y.values()) == N)
    terms, coeffs, const = [], [], 0
    x_at = defaultdict(list)
    for h, sites_h in pd.Series(ps).groupby(ph):
        sites_h = sites_h.to_numpy()
        w = int(C.share_int[h])
        free = sites_h[~constrained[sites_h]]
        if is_op[free].any():               # an existing store that cannot hit capacity covers this hex
            const += w
            continue
        row = []
        if len(free):
            covered = model.NewBoolVar("")
            model.AddBoolOr([y[int(s)] for s in free]).OnlyEnforceIf(covered)
            row.append(covered)
        for s in sites_h[constrained[sites_h]]:
            xv = model.NewBoolVar("")
            if is_new[s]:
                model.AddImplication(xv, y[int(s)])
            x_at[int(s)].append((xv, w))
            row.append(xv)
        if len(row) > 1:
            model.AddAtMostOne(row)
        terms += row
        coeffs += [w] * len(row)
    for s, lst in x_at.items():
        used = cp_model.LinearExpr.WeightedSum([v for v, _ in lst], [w for _, w in lst])
        if is_new[s]:
            model.Add(used <= C.cap_int * y[s])      # the one change: a closed site serves nothing
        else:
            model.Add(used <= C.cap_int)
    model.Maximize(cp_model.LinearExpr.WeightedSum(terms, coeffs) + const)
    t1 = time.perf_counter()
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = max(0.1, time_limit - (t1 - t0))   # the build is slower under load
    solver.parameters.num_workers = WORKERS
    solver.parameters.random_seed = SOLVER_SEED
    status = solver.Solve(model)
    t2 = time.perf_counter()
    ok = status in (cp_model.OPTIMAL, cp_model.FEASIBLE)
    return {"status": solver.StatusName(status), "chosen": sorted(s for s in y if solver.Value(y[s])) if ok else [],
            "obj": solver.ObjectiveValue() / SCALE, "bound": solver.BestObjectiveBound() / SCALE,
            "pairs": len(ps), "new_sites": len(new), "hexes": int(len(np.unique(ph))),
            "constrained sites": int(constrained[np.unique(ps)].sum()),
            "build_s": t1 - t0, "solve_s": t2 - t1, "total_s": t2 - t0}


In [ ]:
tight_runs, tight_rows = {}, []
for name, case in [("res 9 X=30 placeholder 3/4/3", placeholder_9),
                   ("res 9 X=30 sized to the area 17/23/17", sized_9),
                   ("res 8 X=30 placeholder 3/4/3", placeholder_8)]:
    tight_runs[name] = timed_runs(case, N_AREA, solver=solve_plan_tight)
    tight_rows.append(summary(name, case, tight_runs[name], check="never"))

# stress case, not a default scenario and not part of the verdict: the main network at a capacity of 1,000 orders a day
stress_9 = with_network(area_9, placeholder_9.stores, cap_int=round(1000 * SCALE))
runs_stress = timed_runs(stress_9, N_AREA, solver=solve_plan_tight, repeats=1)
row_stress = dict(summary("STRESS res 9 X=30 placeholder, capacity 1,000", stress_9, runs_stress, check="never"), capacity=1000)

cols = ["status", "objective (orders/day)", "bound", "gap % (median)", "constrained sites", "new sites", "pairs",
        "build s (median)", "solve s (median)", "total s median", "min", "max", "all totals", "load before"]
display(pd.DataFrame(tight_rows + [row_stress]).set_index("network")[cols].T)

# correctness against the 0.5 model's results above
by_name = dict(zip(tight_runs, tight_rows))
for name, ref in [("res 9 X=30 sized to the area 17/23/17", res_sized), ("res 8 X=30 placeholder 3/4/3", res_8)]:
    row = by_name[name]
    if row["status"] == "OPTIMAL":
        assert abs(row["objective (orders/day)"] - ref["objective (orders/day)"]) < 0.006, (name, row["objective (orders/day)"], ref["objective (orders/day)"])
        print(f"{name}: OPTIMAL, objective {row['objective (orders/day)']} equals the 0.5 model's {ref['objective (orders/day)']}")
    else:
        print(f"{name}: {row['status']}, objective {row['objective (orders/day)']} against the 0.5 model's {ref['objective (orders/day)']}")
for x in tight_runs["res 9 X=30 placeholder 3/4/3"]:
    assert x["bound"] >= res_main["objective (orders/day)"] - 0.01   # a valid upper bound cannot fall below a plan the 0.5 model found

# the verdict: every default-scenario run within the bar, with the gap stated
default_runs = [x for runs in tight_runs.values() for x in runs]
worst = max(x["total_s"] for x in default_runs)
print(f"default-scenario runs: {len(default_runs)}; slowest total {worst:.2f} s against {SOLVE_BAR_S} s; met: {worst <= SOLVE_BAR_S}")
for name, runs in tight_runs.items():
    gaps = [100 * (x["bound"] - x["obj"]) / x["obj"] if x["obj"] > 0 else np.nan for x in runs]   # nan: no plan found
    print(f"  {name}: gaps % {', '.join(f'{g:.3f}' for g in gaps)} (median {np.median(gaps):.3f}); statuses {[x['status'] for x in runs]}")
print("load average at end (1, 5, 15 min): " + ", ".join(f"{v:.2f}" for v in os.getloadavg()))


**Check: the chosen stores, scored by `evaluate`.** One-off, main case: run 1's plan scored with a 120 s limit (it took
92 s in the experiment). The score must be at least the plan's objective, since the objective is one feasible assignment
for those stores and `evaluate` finds the best one; a difference over 0.01 means the 9 s plan's own assignment was not
the best one for its stores. If `evaluate` cannot prove optimality in 120 s the check says so.


In [ ]:
r_ev = tight_runs["res 9 X=30 placeholder 3/4/3"][0]
t_ev = time.perf_counter()
try:
    _, served_int = evaluate(placeholder_9, list(placeholder_9.stores["Operator"]) + r_ev["chosen"], time_limit=120)
except AssertionError as e:   # only evaluate's own check that its solve reached OPTIMAL
    served_int = None
    print(f"evaluate could not prove optimality in 120 s ({e}) after {time.perf_counter() - t_ev:.1f} s")
if served_int is not None:
    ev = served_int / SCALE
    print(f"evaluate (120 s limit) serves {ev:.2f} for run 1's stores; CP-SAT objective {r_ev['obj']:.2f}; "
          f"difference {ev - r_ev['obj']:+.3f}; {time.perf_counter() - t_ev:.1f} s")
    assert ev >= r_ev["obj"] - 0.005, (ev, r_ev["obj"])   # the stores serve at least what the solver's assignment does

Results from the run on 2026-10-30 (copied here because outputs are stripped on commit). The run covered the tighter
model's cells against the 0.5 model's results above (their runs were not repeated), on the Apple M1, 8 cores, 8 GB, OR-tools
9.15.6755, 8 workers, solver seed 1, 9 s limit. **The machine was not quiet:** another project's job held about three
cores for the whole session, and the 1-minute load average was 6 to 11 before the placeholder-network runs and 12 to 17
before the rest. The placeholder-network plans below are load-sensitive (see the last paragraph).

| default scenario, N = 10, X = 30 | res 9, placeholder 3/4/3 | res 9, sized to the area 17/23/17 | res 8, placeholder 3/4/3 |
|---|---:|---:|---:|
| status | FEASIBLE (9 s limit) | OPTIMAL | OPTIMAL |
| total time, median (min / max) | **9.6 s** (9.5 / 9.7) | **5.3 s** (4.3 / 5.6) | **2.2 s** (1.8 / 2.2) |
| build / solve, median | 0.4 s / 9.3 s | 0.2 s / 5.1 s | 0.02 s / 2.2 s |
| objective, orders a day | 24,329 (runs: 24,329, 22,842, 24,213) | 30,401.05 | 26,619.37 |
| bound | 28,292 | 30,401.05 | 26,619.37 |
| **gap**, bound over objective: runs, median | 16.3%, 23.9%, 16.8%; **16.8%** | 0 | 0 |
| constrained sites | 123 | 22 | 18 |
| load before the runs | 5.8, 8.6, 10.6 | 12.5, 14.5, 15.2 | 16.4, 16.4, 16.6 |

**Stress case (not a default scenario, not in the verdict):** the main network at a capacity of 1,000 orders a day,
one run: FEASIBLE, 10.65 s in total (build 1.2 s), objective 10,632 against a bound of 60,320 (the gap is 467%), 916
constrained sites, load 16.6. At this capacity 9 s gets no usable plan.

**Correctness.** Where the tight model is OPTIMAL its objective equals the 0.5 model's (asserted): 30,401.05 for the
sized network and 26,619.37 for res 8. On the main network the tight bound (28,292) is above the best plan the 0.5
model found (28,266), as it must be. The check of the chosen stores with `evaluate` and a 120 s limit: run 1's stores
(objective 24,329.36) score **24,388.22**, proven optimal, in 5.3 s, so the score is 58.86 orders a day (0.24%) above
the solver's objective, not equal to it. That is consistent (the objective is one feasible assignment for those stores,
and a time-limited run need not have found the best one) but it does not match to 0.01: the objective of a plan cut off
at 9 s understates what its stores serve. `evaluate` was fast here because these stores are a poor plan; it took 92 s
for the 0.5 model's best stores in the earlier experiment.

**Verdict.** The 0.5 model misses the bar on the placeholder network, as measured above (60 s, unproven, a bound 18%
above its plan). **Under the planner's definition (a plan returned within 10 s with its proven gap stated), the tighter
model meets it:** all nine default-scenario runs returned in at most 9.7 s (res 9 placeholder 9.6 s median, res 9 sized
5.3 s, res 8 2.2 s). The stated gaps differ: the sized network and res 8 are proven optimal (gap 0), and the
placeholder network's plan is **16.8% from its bound** (median; 16.3% to 23.9%). That is much looser than the 0.1% a
separate experiment saw at a 10 s limit and a load of 12 to 14 on the same network (objective 28,283 against a bound of
28,292), and even the 0.5 model's 60 s plan (28,266) beats this 9 s plan. Earlier executions of these cells at higher
loads, one above 150 because of another job, gave median gaps of 10% to 11% for the placeholder network, so the range
over four executions is 10% to 17% and the likely cause is CPU contention. A diagnostic run of the
same model (2026-10-30, another project's job holding four of the eight cores, load 10 to 17) did not even reach the
root bound in 9 s (bound 34,478, not 28,292; plans 21,161 to 23,895, 30% to 60% from the bound), and 4 workers did no
better than 8: the 9 s plan's quality depends on the CPU free. The placeholder network's gap is therefore the number to re-measure on a quiet machine; the time
(under 10 s) did not vary that way. Res 9 and X = 30 stand.

**The time limit became a budget.** These runs used a fixed 9 s solver limit. A later run at a load of 38 to 54
(2026-10-30) took up to 10.10 s in total, because the build grew to about 1 s, so `solve_plan_tight` now gives CP-SAT
`SOLVE_LIMIT_S` minus the build time. At a load of 26 to 48 all nine default-scenario runs then took at most 9.10 s,
with poor plans at that load (placeholder network median gap 72%, the sized network 0.4%, res 8 proven optimal; the
capacity-1,000 stress case found no plan). The table above is from the fixed-limit run and was not repeated on a quiet
machine.

## Spot check: the routes and the lookup sheet

About 20 candidate site → hex pairs from the full-area travel-time table, to be timed by hand in Google Maps and compared
with the model ([ADR 0002](../docs/adr/0002-spot-check-times-in-repo.md)). `spotcheck.draw_routes` draws them with a
fixed seed (16), from pairs between 2 and 17 free-flow minutes:

- **5 former bad snaps:** pairs from the out-and-back sample above whose one-snap route rode more than 300 m out and back
  at an end. These are the known bad snaps; whether the fastest of four fixed them is for the check to show.
- **15 others,** 3 from each free-flow band of 2-5, 5-8, 8-11, 11-14 and 14-17 min, so the check covers the short trips the
  default promise depends on as well as the long ones.

`source` is the candidate site's snapped point and `destination` the hex centre, both as `lat,lon` to paste into Google
Maps. The hex centre may sit off a road; Google Maps snaps it to a road of its own choosing, as OSRM did, so the two may
start the last stretch from different roads. `model_min` is free-flow × 1.4, the default congestion multiplier (no pick-and-pack: the app times the ride only).
The sheet is `docs/spot-check.csv`, written once. A re-run draws the same routes and checks the file still holds them; it
never overwrites the app times.

In [ ]:
from catchment.spotcheck import draw_routes, write_sheet

SHEET = DATA.parent / "docs" / "spot-check.csv"
bad = samples.loc[samples["before path m"] > 300, ["candidate", "hex"]]   # one-snap route out and back > 300 m at an end
routes = draw_routes(table_cmd, cand_cmd, demand, bad)
print(f"{len(bad)} known bad snaps in the sample; drawn {int(routes.bad_snap_before.sum())} of them")
print(SHEET.relative_to(DATA.parent), "written" if write_sheet(routes, SHEET) else "already holds these routes; left as is")
display(routes)

### How to do the lookups (#17)

For the person (or the browser agent the planner runs) filling in `docs/spot-check.csv`, one route at a time:

1. **When:** a weekday (Monday to Friday), between 7 and 8 pm India time. Do every route in that window; split the routes
   over more than one evening if one hour is not enough.
2. **Where:** Google Maps, two-wheeler mode. Paste `source` as the start and `destination` as the end (each is
   `lat,lon`). Leave the departure as "Leave now".
3. **What to record:** the typical time Google Maps shows for its first suggested route, in whole minutes, in `app_min`;
   `Google Maps two-wheeler` in `app`; the date and time of the lookup in `looked_up` (for example `2026-11-03 19:20`).
4. **Store only the minutes.** No screenshots, no saved routes, no other details from the app, no bulk collection and no
   API: one route at a time, as a person would. Leave every other column as it is.

If the app finds no two-wheeler route, leave `app_min` empty and fill in `app` and `looked_up`.

**Done** on Friday 30 October 2026, 19:00-19:07, by a browser agent the planner launched, as ADR 0002 allows: one route
at a time, the `lat,lon` pair opened in two-wheeler mode with "Leave now", the time shown for the first route recorded.
Under "Leave now" that is the live estimate for the moment of the lookup, not a typical range. Every route found a
two-wheeler route. Only `app_min`, `app` and `looked_up` were written.

## Spot check: the error (#18)

The ratio is the app time ÷ the modelled time, where modelled is free-flow × 1.4 (the default congestion multiplier). Below, its median and spread over the routes with an app time, the five worst routes, and the implied multiplier (app ÷ free-flow) against the 1.4 default. The split by former bad snap shows whether those routes stand out, and where they rank.

The app times are whole minutes (±0.5 min, large against the short routes) and are "Leave now" live estimates on a Friday 19:00-19:07, not a typical range.


In [ ]:
from catchment.spotcheck import compare

sheet = pd.read_csv(SHEET, dtype={"candidate": str, "hex": str})
errors, summary = compare(sheet)
print(pd.Series(summary).to_string())
display(errors.head(5)[["route", "free_flow_min", "model_min", "app_min", "ratio", "implied", "bad_snap_before"]])

Results from the run on 2026-10-31 (copied here because outputs are stripped on commit). 20 routes, every one with an app
time; Google Maps two-wheeler, "Leave now" on Friday 30 October 2026, 19:00-19:07, so live evening-peak estimates, not
a typical range.

| | median | 25th-75th percentile | min / max |
|---|---:|---:|---:|
| **error:** app ÷ modelled (free-flow × 1.4) | **1.28** | 1.17-1.54 | 0.97 / 2.65 |
| **implied multiplier:** app ÷ free-flow | **1.79** | 1.64-2.16 | 1.35 / 3.70 |

**The model is optimistic: the app took 28% longer than free-flow × 1.4 on the median route**, and longer on 19 of 20
(the exception, route 4, is 3% under). The median gap is 3.7 min. The worst routes:

| route | free-flow min | modelled min | app min | app ÷ modelled | former bad snap |
|---:|---:|---:|---:|---:|:---:|
| 1 | 2.97 | 4.16 | 11 | 2.65 | yes |
| 2 | 3.24 | 4.54 | 10 | 2.20 | |
| 16 | 11.74 | 16.44 | 29 | 1.76 | |
| 5 | 4.46 | 6.24 | 11 | 1.76 | |
| 20 | 16.89 | 23.65 | 37 | 1.56 | |

**The implied multiplier is about 1.8 against the 1.4 default**, and it is not a fixed overhead at the ends of short
trips: a straight-line fit gives app ≈ 0.75 + 1.82 × free-flow min, and the median per free-flow band of the draw (the
15 routes that were not bad snaps, three per band) is 2.47 at 2-5 min, then 1.68, 1.78, 1.80 and 1.96 at 5-8, 8-11,
11-14 and 14-17 min. The short band is the noisiest, since a whole
minute is a large share of a 3-minute trip. At 1.8 the default promise (10 min, 3 min pick-and-pack) reaches 3.9
free-flow minutes instead of 5.0. Whether the default should move is a decision for the planner, not made here:
these are one Friday evening's live estimates, the evening peak the 7-8 pm window was chosen for, and 1.8 sits inside
the lever's 1.0-2.0 range.

**The former bad snaps no longer stand out as a group:** median ratio 1.33 against 1.28 for the other 15. They rank
1, 7, 9, 14 and 16 of 20 (1 = worst). Route 1 is the worst route overall (11 min in the app against 4.2 modelled), so
one fastest-of-four snap may still be wrong, or the short trip's whole-minute rounding and the app's own snap of the
hex centre add up; one route cannot tell which.


## Next

Stage 1 is done (see the roadmap). Stage 2 moves `02-plan` and the tighter model into `src/catchment/`, with the
synthetic network generator.
